In [3]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight

# =========================
# CONFIG
# =========================
DATASET_ROOT = "/home/feliciano/Downloads/Preliminary Data/AllFish_2secSplit"
SR = 22050
DURATION = 2.0
TARGET_LEN = int(SR * DURATION)

N_MELS = 64
HOP_LENGTH = 512
N_FFT = 1024

BATCH_SIZE = 16
EPOCHS = 12
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# =========================
# AUDIO + FEATURES
# =========================
def load_audio(path):
    y, _ = librosa.load(path, sr=SR)
    if len(y) < TARGET_LEN:
        y = np.pad(y, (0, TARGET_LEN - len(y)))
    return y[:TARGET_LEN]

def log_mel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # 🔥 IMPORTANT: per-sample normalization
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db

# =========================
# DATASET
# =========================
class FishDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        y = load_audio(self.files[idx])
        mel = log_mel(y)
        mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)  # (1, 64, T)
        return mel, torch.tensor(self.labels[idx], dtype=torch.long)

# =========================
# AST-LIKE MODEL
# =========================
class PatchEmbed(nn.Module):
    def __init__(self, in_ch=1, emb_dim=256, patch_size=(16, 16)):
        super().__init__()
        self.proj = nn.Conv2d(
            in_ch, emb_dim, kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)                 # (B, C, H, W)
        x = x.flatten(2).transpose(1, 2) # (B, N, C)
        return x

class ASTLike(nn.Module):
    def __init__(
        self,
        num_classes,
        emb_dim=256,
        depth=6,
        heads=8,
        mlp_dim=512,
        patch_size=(16, 16),
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(1, emb_dim, patch_size)
        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_dim))
        self.pos_embed = None

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=emb_dim,
                nhead=heads,
                dim_feedforward=mlp_dim,
                batch_first=True  # 🔥 FIX
            ),
            num_layers=depth
        )

        self.fc = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)  # (B, N, C)

        if self.pos_embed is None or self.pos_embed.shape[1] != x.shape[1] + 1:
            self.pos_embed = nn.Parameter(
                torch.randn(1, x.shape[1] + 1, x.shape[2], device=x.device)
            )

        cls_tokens = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed

        x = self.transformer(x)
        x = x[:, 0]  # CLS token

        return self.fc(x)

# =========================
# LOAD FILES
# =========================
files, labels = [], []
for cls in sorted(os.listdir(DATASET_ROOT)):
    d = os.path.join(DATASET_ROOT, cls)
    if not os.path.isdir(d):
        continue
    for f in os.listdir(d):
        if f.endswith(".wav"):
            files.append(os.path.join(d, f))
            labels.append(cls)

le = LabelEncoder()
labels = le.fit_transform(labels)
NUM_CLASSES = len(le.classes_)

# =========================
# CLASS WEIGHTS (CRITICAL)
# =========================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

# =========================
# SPLIT 70 / 20 / 10
# =========================
f_train, f_tmp, y_train, y_tmp = train_test_split(
    files, labels, test_size=0.30, stratify=labels, random_state=SEED
)

f_test, f_val, y_test, y_val = train_test_split(
    f_tmp, y_tmp, test_size=1/3, stratify=y_tmp, random_state=SEED
)

train_dl = DataLoader(FishDataset(f_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(FishDataset(f_val, y_val), batch_size=BATCH_SIZE)
test_dl  = DataLoader(FishDataset(f_test, y_test), batch_size=BATCH_SIZE)

# =========================
# EVALUATION
# =========================
def evaluate(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    if np.any(np.isnan(y_prob)):
        y_prob = np.nan_to_num(y_prob)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro"),
        "f1": f1_score(y_true, y_pred, average="macro"),
        "auc": roc_auc_score(y_true, y_prob, multi_class="ovo"),
    }

# =========================
# TRAIN
# =========================
model = ASTLike(NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4  # 🔥 lower LR for transformer
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

for epoch in range(EPOCHS):
    model.train()
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    val_metrics = evaluate(model, val_dl)
    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"VAL Acc: {val_metrics['accuracy']:.4f} | "
        f"VAL AUC: {val_metrics['auc']:.4f}"
    )

# =========================
# TEST
# =========================
test_metrics = evaluate(model, test_dl)
print("\n=== AST-Like Transformer Test Metrics ===")
print(test_metrics)


Epoch 1/12 | VAL Acc: 0.6750 | VAL AUC: 0.4816
Epoch 2/12 | VAL Acc: 0.6741 | VAL AUC: 0.8202
Epoch 3/12 | VAL Acc: 0.6741 | VAL AUC: 0.5055
Epoch 4/12 | VAL Acc: 0.6741 | VAL AUC: 0.5002
Epoch 5/12 | VAL Acc: 0.6741 | VAL AUC: 0.4934
Epoch 6/12 | VAL Acc: 0.6741 | VAL AUC: 0.5000
Epoch 7/12 | VAL Acc: 0.6741 | VAL AUC: 0.5004
Epoch 8/12 | VAL Acc: 0.6741 | VAL AUC: 0.4998
Epoch 9/12 | VAL Acc: 0.6741 | VAL AUC: 0.4961
Epoch 10/12 | VAL Acc: 0.6741 | VAL AUC: 0.4988
Epoch 11/12 | VAL Acc: 0.6741 | VAL AUC: 0.4848
Epoch 12/12 | VAL Acc: 0.6741 | VAL AUC: 0.5016

=== AST-Like Transformer Test Metrics ===
{'accuracy': 0.6741449062155204, 'precision': 0.1685362265538801, 'recall': 0.25, 'f1': 0.20134007029876977, 'auc': 0.5025541641067671}
